# Whisper Fine-Tuning — Duration Sweep
--------

This notebook runs a series of fine-tuning experiments on progressively different datasets as shown in the table below:


| Section | Dataset | Notes |
|---------|---------|-------|
| A | Full 14h (no filtering) | Baseline run | 



## 1. Python Setup

In [1]:
ENV = "local"  # options: "local", "colab"

In [2]:
import os
import time
from datetime import datetime
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import DatasetDict
import wandb

# Project path   
if ENV == "jupyter-hub":
    PROJECT_SRC = Path('/home/jupyter-wb344850/chichewa-asr')
else:
    # Modify index at the end as required to point to the parent directory of `src/` (which contains the `src` package)
    PROJECT_SRC = Path().cwd().parents[1]

sys.path.append(str(PROJECT_SRC))

from src.train.train_whisper import load_config
from src.train.whisper_duration_experiment import (
    load_model_and_processor,
    prepare_train_dataset,
    prepare_test_dataset,
    run_training,
    run_evaluation,
)

/Users/dmatekenya/git-repos/chichewa-asr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING Disabling SSL verification.  Connections to this server are not verified and may be insecure!


## 2. Paths and Global Configuration

In [3]:
# ==========================================
# 1. BASE WORKING DIRECTORY AND PATHS
# ==========================================
# Base directory of the project (one level up from src/)
if ENV == "jupyter-hub":
    DIR_BASE = Path('/home/jupyter-wb344850/chichewa-asr')
else:
    DIR_BASE = Path().cwd().parents[1]

DIR_DATA   = DIR_BASE / "data"
# Hold out test set 
DIR_TEST               = DIR_DATA / "test"
FILE_MANIFEST_TEST     = DIR_TEST / "metadata.csv"

# Audio files for the dev set and the dev set with nested duration
DIR_DEV                     = DIR_DATA / "dev"
FILE_MANIFEST_DEV = DIR_DEV / "metadata.csv"
DIR_DEV_NESTED_DURATION     = DIR_DATA / "dev_nested_duration"

# Hyperparameter config file path
FILE_CONFIG = DIR_BASE / "configs" / "whisper_hparams_baseline.yaml"
FILE_CONFIG_DEBUG = DIR_BASE / "configs" / "whisper_hparams_debug.yaml"

# Outputs directory where we keep the results of the experiments
DIR_OUTPUTS = DIR_BASE / "outputs"
DIR_RESULTS = DIR_OUTPUTS / "datasets_experiments"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

# Model checkpoint directory (for saving model checkpoints during training)
DIR_MODELS = DIR_BASE / "models"
DIR_WANDB_LOGS = DIR_MODELS / "wandb"
DIR_MODEL_CHECKPOINTS = DIR_MODELS / "checkpoints"
DIR_MODEL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# ==========================================
# 2. HARDWARE SETUP
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
# Human-readable label for the machine/device used — update as needed
DEVICE_LABEL = "mps-96gb-ram" 

# ==========================================
# 3. DEBUGGING SETTINGS
# ==========================================
DEBUG = True

Using device: cpu


In [4]:
# ==========================================
# 3. LOGIN TO HUGGINGFACE HUB AND WANDB
# ==========================================
# Load .env file
load_dotenv()

# Log into Huggingface Hub using token from .env file
login(token=os.getenv("HF_TOKEN"))

# Lof into Weights & Biases using API key from .env file
os.environ["WANDB_DIR"] = str(DIR_WANDB_LOGS)  # Set the directory where wandb logs will be stored
wandb.login(key=os.getenv("WANDB_API_KEY"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/dmatekenya/.netrc
wandb: Currently logged in as: dmatekenya (dmatekenya-the-world-bank-group) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 4. Model Configurations and Held-Out Test Set Preparation 

These two objects are created once and shared across every duration run.
The test set is preprocessed with a temporary processor (base model weights);
audio features are model-agnostic so this is correct for all runs.

In [5]:
# ============================
# 1. LOAD BASE CONFIG ONCE
# ============================
if DEBUG:
    print("\nDEBUG MODE: ON - Using debug config with minimal training steps and batch sizes")
    FILE_CONFIG = FILE_CONFIG_DEBUG

base_config = load_config(FILE_CONFIG)
print("Config loaded:", FILE_CONFIG)

# =========================================
# 2. MODEL AND HUB SETTINGS
# =========================================
MODEL_ID     = base_config["model"]["model_name_or_path"]  
MODEL_NAME   = MODEL_ID.split("/")[-1]                     
BASE_HUB_MODEL_ID = f"dmatekenya/{MODEL_NAME}-chichewa"  


DEBUG MODE: ON - Using debug config with minimal training steps and batch sizes
Config loaded: /Users/dmatekenya/git-repos/chichewa-asr/configs/whisper_hparams_debug.yaml


In [6]:
 # =========================================
# 3. PREPARE HELD-OUT TEST SET
# =========================================
dataset_test = prepare_test_dataset(
    FILE_MANIFEST_TEST,
    audio_dir=DIR_TEST,
    base_config=base_config,
    audio_fname_col="audio_filename",
    duration_col="duration_seconds",
)

print(f"\nHeld-out test set ready: {len(dataset_test):,} utterances")


  Loading test data: /Users/dmatekenya/git-repos/chichewa-asr/data/test/metadata.csv
Total duration : 1.68 hrs  (573 utterances)
  all_data   :   573 utterances  |  1.68 hrs (100.0%)


Map (num_proc=1): 100%|██████████| 573/573 [00:11<00:00, 49.57 examples/s]


Held-out test set ready: 573 utterances


## 5. Run Experiments

### 5.1 Baseline Dataset
We run on the whole 14 hours without any filters 

In [7]:
# ==========================================
# 1. MODEL NAME AND HUB SETTINGS
# ==========================================    
# Set duration label for this run
duration_label = "14h"
# Hub model ID for this run and output dir for this run 
hub_model_id = f"{BASE_HUB_MODEL_ID}-{duration_label}"
output_dir = DIR_MODEL_CHECKPOINTS / f"{MODEL_NAME}-chichewa-{duration_label}"

# ==========================================
# 2. LOAD MODEL AND PROCESSOR
# ==========================================  
model, processor = load_model_and_processor(base_config)

# ==========================================
# 3. PREPARE TRAINING DATA
# ========================================== 
dataset_train = prepare_train_dataset(FILE_MANIFEST_DEV, DIR_DEV, processor)
if DEBUG:
    print("\nDEBUG MODE: Using a small subset of the training data for quick iterations")
    dataset_train = DatasetDict({
    "train":      dataset_train["train"].select(range(100)),
    "validation": dataset_train["validation"].select(range(20)),
})


# ==========================================
# 4. TRAIN MODEL
# ==========================================
print(f"\n{'='*60}\n  EXPERIMENT: {duration_label}\n{'='*60}")
train_start = time.time()
trainer = run_training(model, processor, dataset_train, base_config, hub_model_id, output_dir)
train_minutes = (time.time() - train_start) / 60

# ==========================================
# 5. PUSH TO HUB (OPTIONAL)
# ==========================================
print(f"  Pushing to Hub: {hub_model_id}")
trainer.push_to_hub()

# ==========================================
# 6. EVALUATE ON HELD-OUT TEST SET
# ==========================================
df_res_14h = run_evaluation(
    model,
    processor,
    dataset_test,
    duration_label,
    DIR_RESULTS,
    model_id=hub_model_id,
    debug=DEBUG,
)

  Loading Whisper model: openai/whisper-small


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 9796.29it/s]


  Loading train data: /Users/dmatekenya/git-repos/chichewa-asr/data/dev/metadata.csv
Total duration : 13.85 hrs  (3,870 utterances)
  train       : 3,484 utterances  |  12.46 hrs  (90.0%)
  validation  :   386 utterances  |  1.39 hrs  (10.0%)


Map (num_proc=4): 100%|██████████| 386/386 [00:03<00:00, 99.54 examples/s] 



DEBUG MODE: Using a small subset of the training data for quick iterations

  EXPERIMENT: 14h
  Training ...


...

Step,Training Loss,Validation Loss,Wer,Cer
10,4.199762,3.747340,148.785872,110.877404
20,4.423598,3.536222,155.408389,128.185096


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  Pushing to Hub: dmatekenya/whisper-small-chichewa-14h


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]
Processing Files (2 / 2): 100%|██████████|  967MB /  967MB, 8.37MB/s  
New Data Upload: 100%|██████████|  966MB /  966MB, 8.37MB/s  


[DEBUG] Running evaluation on a small sample of the test set.
  WER (corpus): 106.12%   CER (corpus): 48.33%
  Predictions saved: /Users/dmatekenya/git-repos/chichewa-asr/outputs/datasets_experiments/predictions_14h.csv


In [8]:
df_res_14h.head()

,model_id,audio_fname,reference,prediction,wer_utterance,wer_avg,cer_avg
0,dmatekenya/whisper-small-chichewa-14h,priscilla_self_records46.wav,Kodi zikhadabo amaika bwanji amene akudziwa an...,kodiji kada bamaikabonja meniakujiwa antitandizi,100.0,106.122449,48.326055
1,dmatekenya/whisper-small-chichewa-14h,priscilla_self_records414.wav,Akuti adavulala kwambiri anavulallira mkati,Aukuta davula lakwambi iti anavula idankati.,120.0,106.122449,48.326055
2,dmatekenya/whisper-small-chichewa-14h,priscilla_self_records418.wav,Mtsikana uyu amaphoda kwambiri,ntikonu nwiyama poda wanbe ee liye,150.0,106.122449,48.326055
3,dmatekenya/whisper-small-chichewa-14h,priscilla_self_records211.wav,Kaodeni katundu ku Kampepuza ku Ntcheu.,"Aode ni garundrubu kampevusa, uncheu.",100.0,106.122449,48.326055
4,dmatekenya/whisper-small-chichewa-14h,AU40.wav,Tazimitsako magetsiwo Tazimitsako magetsiwo,Dazi mitzaku magesio,100.0,106.122449,48.326055
